# FDRL-IDS: Federated Deep Reinforcement Learning based IDS

**Paper:** "Federated reinforcement learning based intrusion detection system using dynamic attention mechanism"  
**Journal:** Journal of Information Security and Applications 78 (2023) 103608

---

### RL Algorithms

| Algorithm             | Description                                    | Reference                                         |
| --------------------- | ---------------------------------------------- | ------------------------------------------------- |
| **DQN** (original)    | Deep Q-Network with PER                        | Paper Algorithm 1, Page 6                         |
| **PPO** (improvement) | Proximal Policy Optimization + Advanced Reward | "Deep RL for Wireless" Algorithm 3.8, Pages 72-73 |

### Research Scenarios

| Scenario                 | Train Dataset | Test Dataset          | Purpose                         |
| ------------------------ | ------------- | --------------------- | ------------------------------- |
| **A (Baseline)**         | NSL-KDD       | NSL-KDD               | Reproduce paper results         |
| **B (Same-domain IoT)**  | CIC-IoMT-2024 | CIC-IoMT-2024         | Architecture on modern IoT data |
| **C (Cross-domain IoT)** | CIC-IoMT-2024 | CIC-Edge-IIoTSet-2022 | Cross-domain generalization     |

### Required Kaggle Datasets

- **Scenario A:** Add `nsl-kdd` dataset
- **Scenario B & C:** Add `cic-bccc-nrc-iomt-2024` and `cic-bccc-nrc-edge-iiotset-2022` datasets
- Upload project source code as dataset `fdrl-ids` (zip containing src/ and main_train.py)


In [ ]:
# ============================================================
# Cell 1 - Setup: install deps & clone source code from GitHub
# ============================================================
import os, sys, shutil

WORKING = '/kaggle/working/NT549'

# ---------- Option 1: Clone from GitHub (recommended) ----------
# Change this to your GitHub repo URL
GITHUB_REPO = 'https://github.com/namanh2508/NT549.git'

if not os.path.isdir(WORKING):
    os.system(f'git clone {GITHUB_REPO} {WORKING}')
    print(f'Cloned from {GITHUB_REPO}')
else:
    # Pull latest changes if already cloned
    os.system(f'cd {WORKING} && git pull')
    print(f'Pulled latest changes')

# ---------- Option 2: Fallback to uploaded dataset ----------
if not os.path.isfile(os.path.join(WORKING, 'main_train.py')):
    PROJECT_CANDIDATES = [
        '/kaggle/input/fdrl-ids',
        '/kaggle/input/fdrl-ids/NT549',
        '/kaggle/input/nt549',
        '/kaggle/input/nt549/NT549',
    ]
    project_root = None
    for cand in PROJECT_CANDIDATES:
        if os.path.isfile(os.path.join(cand, 'main_train.py')):
            project_root = cand
            break
    if project_root:
        os.makedirs(WORKING, exist_ok=True)
        for item in ['src', 'main_train.py']:
            s = os.path.join(project_root, item)
            d = os.path.join(WORKING, item)
            if os.path.exists(s) and not os.path.exists(d):
                if os.path.isdir(s):
                    shutil.copytree(s, d)
                else:
                    shutil.copy2(s, d)
        print(f'Copied project from {project_root}')

os.chdir(WORKING)
sys.path.insert(0, WORKING)

!pip install -q torch numpy pandas scikit-learn matplotlib
print('Setup complete.')

In [ ]:
# ============================================================
# Cell 2 - SCENARIO & MODEL CONFIGURATION
# ============================================================
# Change SCENARIO to switch between experiments:
#   'A' = NSL-KDD baseline (reproduce paper)
#   'B' = Same-domain IoT (CIC-IoMT-2024 train & test)
#   'C' = Cross-domain IoT (CIC-IoMT-2024 train -> CIC-Edge-IIoTSet-2022 test)

SCENARIO = 'A'  # <-- CHANGE THIS: 'A', 'B', or 'C'

# ============================================================
# AGENT TYPE: 'dqn' (original paper) or 'ppo' (improvement)
# ============================================================
# 'dqn' = Deep Q-Network (Algorithm 1, bai bao goc)
# 'ppo' = Proximal Policy Optimization (cai tien, Algorithm 3.8 - PPO book)
AGENT_TYPE = 'ppo'  # <-- CHANGE THIS: 'dqn' or 'ppo'

# ============================================================
# AGGREGATION METHOD (FL Improvements)
# ============================================================
# Options:
#   'attention'                    - Original paper (Algorithm 2)
#   'fltrust'                      - FLTrust Byzantine-resilient
#   'attention_fltrust'            - Dynamic Attention + FLTrust combined
#   'fedplus'                      - Fed+ server momentum
#   'fltrust_fedplus'             - FLTrust + Fed+ combined
#   'attention_fltrust_fedplus'    - FULL: Dynamic Attention + FLTrust + Fed+
AGGREGATE_METHOD = 'attention_fltrust_fedplus'  # <-- CHANGE THIS

# Common settings
EXPERIMENT = 'random'        # 'random' or 'customized'
NUM_AGENTS = 8               # 8 for random, 2 for customized
NUM_ROUNDS = 50
EPISODES_PER_ROUND = 3
USE_DAE = True
SEED = 42

# CIC-specific: max rows per CSV file (caps at ~150K total)
SUBSAMPLE_PER_FILE = 10000

# Attention params (will be auto-set for customized)
ATTN_K = 30
ATTN_A = 50

# ============================================================
# FLTrust PARAMETERS (for Byzantine defense)
# ============================================================
FLTRUST_TAU = 0.5       # Cosine similarity threshold (default 0.5)
FLTRUST_BETA = 0.5      # Dimension suppression factor (default 0.5)
FLTRUST_PROXY_RATIO = 0.01  # Proxy data ratio (1% of data)

# ============================================================
# Fed+ PARAMETERS (for non-IID correction)
# ============================================================
FEDPLUS_BETA = 0.9       # Momentum coefficient (default 0.9)
FEDPLUS_ETA = 1.0         # Learning rate (default 1.0)

# ============================================================
# REWARD CONFIGURATION (Improvement 2 - RL Hướng 2)
# ============================================================
# R(t) = alpha*TP - beta*FP - gamma*FN + delta*(1 - latency) + epsilon*novelty + tn*TN
# Chi ap dung khi AGENT_TYPE = 'ppo' (DQN dung reward +/-1 goc)
REWARD_CONFIG = {
    'alpha':       1.0,   # Thuong True Positive (phat hien dung attack)
    'beta':        0.5,   # Phat False Positive (canh bao sai)
    'gamma_fn':    2.0,   # Phat False Negative (bo sot attack) - nang nhat
    'delta':       0.0,   # Latency bonus (0 = tat, dung cho batch mode)
    'epsilon_nov': 0.3,   # Novelty bonus (khuyen khich generalization)
    'tn':          0.2,   # True Negative reward (phan loai dung normal)
}

OUTPUT_DIR = '/kaggle/working/results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Scenario: {SCENARIO}')
print(f'Agent type: {AGENT_TYPE.upper()}')
print(f'Aggregation: {AGGREGATE_METHOD}')
print(f'Experiment: {EXPERIMENT}, Agents: {NUM_AGENTS}, Rounds: {NUM_ROUNDS}')
if AGENT_TYPE == 'ppo':
    print(f'Reward weights: alpha={REWARD_CONFIG["alpha"]}, beta={REWARD_CONFIG["beta"]}, '
          f'gamma={REWARD_CONFIG["gamma_fn"]}, delta={REWARD_CONFIG["delta"]}, '
          f'epsilon={REWARD_CONFIG["epsilon_nov"]}, tn={REWARD_CONFIG["tn"]}')

In [ ]:
# ============================================================
# Cell 3 - Imports & device check
# ============================================================
import numpy as np
import pandas as pd
import torch
import time, json

sys.path.insert(0, os.getcwd())

from src.data.nsl_kdd import (
    load_nsl_kdd, preprocess_nsl_kdd,
    distribute_data_random, distribute_data_customized
)
from src.data.cic_common import (
    load_cic_dataset, preprocess_cic, resolve_cic_dataset_path,
    distribute_data_customized_iomt, distribute_data_customized_iiot
)
from src.models.denoising_autoencoder import (
    DenoisingAutoencoder, train_dae, transform_with_dae
)
from src.federated_learning.orchestrator import FederatedOrchestrator
from src.utils.metrics import compute_metrics, print_metrics, compute_roc_data
from src.utils.visualization import (
    plot_all_training_curves, plot_roc_curve, plot_accuracy_vs_num_agents
)
from src.utils.config import *
from sklearn.preprocessing import MinMaxScaler

np.random.seed(SEED)
torch.manual_seed(SEED)

# --- CUDA compatibility check ---
# Tesla P100 = sm_60, not supported by PyTorch >= 2.1 (requires sm_70+)
# T4 = sm_75, A100 = sm_80 -- both are supported. Switch accelerator if possible.
device = torch.device('cpu')
if torch.cuda.is_available():
    try:
        torch.zeros(1).cuda()           # probes actual kernel execution
        device = torch.device('cuda')
        torch.cuda.manual_seed_all(SEED)
        print(f'GPU: {torch.cuda.get_device_name(0)} (capability sm_{torch.cuda.get_device_capability()[0]}{torch.cuda.get_device_capability()[1]})')
    except Exception as e:
        print(f'[Warning] CUDA not usable: {e}')
        print('[Warning] Falling back to CPU.')
        print('[Tip] Switch Kaggle accelerator from P100 to T4 (sm_75) for GPU support.')

print(f'Device: {device}')

## Step 1 - Load & Preprocess Dataset


In [ ]:
# ============================================================
# Cell 4 - Load and preprocess based on SCENARIO
# ============================================================
# Kaggle dataset structure assumed (from Data Explorer):
#   /kaggle/input/cic-bccc-nrc-iomt-2024/CIC-BCCC-NRC-IoMT-2024/*.csv
#   /kaggle/input/cic-bccc-nrc-edge-iiotset-2022/CIC-BCCC-NRC-Edge-IIoTSet-2022/*.csv
#   /kaggle/input/nsl-kdd/NSL-KDD/KDDTrain+.txt  (or KDDTrain.txt on Kaggle)

train_attack_names = None
test_attack_names = None
raw_train_df = None

if SCENARIO == 'A':
    # --- NSL-KDD ---
    # Kaggle puts files at: /kaggle/input/nsl-kdd/NSL-KDD/KDDTrain+.txt
    NSL_CANDIDATES = [
        '/kaggle/input/datasets/phungvannamanh/nt549-full-datasets/NSL-KDD/NSL-KDD'
    ]
    TRAIN_NAMES = ['KDDTrain+.txt', 'KDDTrain.txt']  # Kaggle strips '+' on upload
    TEST_NAMES  = ['KDDTest+.txt',  'KDDTest.txt']

    train_path = test_path = None
    for cand in NSL_CANDIDATES:
        for tn, tsn in zip(TRAIN_NAMES, TEST_NAMES):
            tp = os.path.join(cand, tn)
            tsp = os.path.join(cand, tsn)
            if os.path.isfile(tp) and os.path.isfile(tsp):
                train_path, test_path = tp, tsp
                break
        if train_path:
            break

    if not train_path:
        raise FileNotFoundError(
            'NSL-KDD not found. Expected:\n'
            '  /kaggle/input/nsl-kdd/NSL-KDD/KDDTrain+.txt (or KDDTrain.txt)\n'
            'Add the nsl-kdd dataset to this notebook.'
        )

    print(f'Train: {train_path}')
    print(f'Test:  {test_path}')
    train_df, test_df = load_nsl_kdd(train_path, test_path)
    raw_train_df = train_df.copy()

    X_train, y_train, X_test, y_test, scaler, feature_dim = preprocess_nsl_kdd(
        train_df, test_df
    )
    dataset_label = 'NSL_KDD'
    test_label = 'NSL_KDD'
    is_cross_dataset = False

elif SCENARIO in ('B', 'C'):
    # --- CIC datasets ---
    # resolve_cic_dataset_path searches nested Kaggle layout:
    #   /kaggle/input/<slug>/<OriginalFolderName>/*.csv
    # For IoMT: /kaggle/input/cic-bccc-nrc-iomt-2024/CIC-BCCC-NRC-IoMT-2024/
    # For IIoT: /kaggle/input/cic-bccc-nrc-edge-iiotset-2022/CIC-BCCC-NRC-Edge-IIoTSet-2022/
    CIC_DATA_DIR = '/kaggle/input/datasets/phungvannamanh/nt549-full-datasets'

    train_dir = resolve_cic_dataset_path(CIC_DATA_DIR, 'iomt')
    print(f'Train dir: {train_dir}')
    train_df = load_cic_dataset(train_dir, subsample_per_file=SUBSAMPLE_PER_FILE, seed=SEED)

    if SCENARIO == 'B':
        # Same-domain: 80/20 split from IoMT
        test_df = None
        dataset_label = 'IOMT'
        test_label = 'IOMT'
        is_cross_dataset = False
    else:  # SCENARIO == 'C'
        # Cross-domain: test on IIoT
        test_dir = resolve_cic_dataset_path(CIC_DATA_DIR, 'iiot')
        print(f'Test dir:  {test_dir}')
        test_df = load_cic_dataset(test_dir, subsample_per_file=SUBSAMPLE_PER_FILE, seed=SEED + 1)
        dataset_label = 'IOMT'
        test_label = 'IIOT'
        is_cross_dataset = True

    result = preprocess_cic(train_df, test_df, seed=SEED)
    X_train, y_train, X_test, y_test = result[0], result[1], result[2], result[3]
    scaler, feature_dim = result[4], result[5]
    train_attack_names = result[6]
    test_attack_names = result[7]

print(f'\nFeature dimension: {feature_dim}')
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Dataset: Train={dataset_label}, Test={test_label}')

## Step 2 - Denoising Autoencoder (Section 4.1, Page 5)


In [ ]:
# ============================================================
# Cell 5 - Train DAE and transform features
# ============================================================
if USE_DAE:
    print('Training Denoising Autoencoder...')
    dae = DenoisingAutoencoder(
        input_dim=feature_dim,
        hidden_dim=DAE_HIDDEN_DIM,
        noise_factor=DAE_NOISE_FACTOR
    )
    dae, dae_losses = train_dae(
        dae, X_train, device,
        epochs=DAE_EPOCHS,
        batch_size=DAE_BATCH_SIZE,
        lr=DAE_LEARNING_RATE
    )
    X_train_dae = transform_with_dae(dae, X_train, device)
    X_test_dae  = transform_with_dae(dae, X_test, device)

    dae_scaler = MinMaxScaler()
    X_train_final = dae_scaler.fit_transform(X_train_dae)
    X_test_final  = dae_scaler.transform(X_test_dae)
    # Warning 3 fix: clip test data after DAE scaler (cross-domain values may exceed [0,1])
    X_test_final = np.clip(X_test_final, 0.0, 1.0)
    input_dim = X_train_final.shape[1]
    print(f'Feature dim after DAE: {input_dim}')
else:
    X_train_final, X_test_final = X_train, X_test
    input_dim = feature_dim
    print(f'Using raw features, dim={input_dim}')

## Step 3 - Distribute Data Among Agents (Section 6, Pages 8-9)


In [ ]:
# ============================================================
# Cell 6 - Data distribution
# ============================================================
if EXPERIMENT == 'random':
    num_agents = NUM_AGENTS
    agent_data = distribute_data_random(X_train_final, y_train, num_agents, SEED)
    attn_k, attn_a = ATTN_K, ATTN_A

elif EXPERIMENT == 'customized':
    num_agents = 2
    if SCENARIO == 'A':
        agent_data = distribute_data_customized(
            raw_train_df, y_train, X_train_final, num_agents=2, seed=SEED
        )
    elif SCENARIO in ('B', 'C'):
        # Bug 4 fix: select customized distribution based on training dataset
        # Scenario B & C both train on IoMT, but if extended to IIoT training,
        # this will correctly dispatch to distribute_data_customized_iiot()
        if dataset_label == 'IIOT':
            agent_data = distribute_data_customized_iiot(
                train_attack_names, y_train, X_train_final, seed=SEED
            )
        else:
            agent_data = distribute_data_customized_iomt(
                train_attack_names, y_train, X_train_final, seed=SEED
            )
    attn_k, attn_a = 50000, 200

print(f'\nExperiment: {EXPERIMENT}, Agents: {num_agents}')
print(f'Attention params: k={attn_k}, a={attn_a}')

## Step 4 - Federated Training (Algorithms 1-3, Section 4, Pages 4-6)


In [ ]:
# ============================================================
# Cell 7 - Create orchestrator & train
# ============================================================
# Xác định learning rate theo loại agent
agent_lr = PPO_LEARNING_RATE if AGENT_TYPE == 'ppo' else DQN_LEARNING_RATE

orchestrator = FederatedOrchestrator(
    num_agents=num_agents,
    input_dim=input_dim,
    hidden_layers=DQN_HIDDEN_LAYERS,
    num_actions=NUM_ACTIONS,
    lr=agent_lr,
    gamma=GAMMA,
    epsilon_start=EPSILON_START,
    epsilon_end=EPSILON_END,
    epsilon_decay=EPSILON_DECAY,
    memory_capacity=MEMORY_CAPACITY,
    batch_size_replay=BATCH_SIZE_REPLAY,
    per_alpha=PER_ALPHA,
    per_beta_start=PER_BETA_START,
    per_beta_end=PER_BETA_END,
    omega=OMEGA,
    dropout=DQN_DROPOUT,
    attention_k=attn_k,
    attention_a=attn_a,
    episodes_per_round=EPISODES_PER_ROUND,
    device=str(device),
    # --- PPO-specific params (ignored when agent_type='dqn') ---
    agent_type=AGENT_TYPE,
    ppo_clip_epsilon=PPO_CLIP_EPSILON,
    ppo_epochs=PPO_EPOCHS,
    ppo_mini_batch_size=PPO_MINI_BATCH_SIZE,
    ppo_value_coef=PPO_VALUE_COEF,
    ppo_entropy_coef=PPO_ENTROPY_COEF,
    ppo_max_grad_norm=PPO_MAX_GRAD_NORM,
    # --- Reward params (chỉ PPO agent dùng advanced reward) ---
    reward_alpha=REWARD_CONFIG['alpha'],
    reward_beta=REWARD_CONFIG['beta'],
    reward_gamma_fn=REWARD_CONFIG['gamma_fn'],
    reward_delta=REWARD_CONFIG['delta'],
    reward_epsilon_nov=REWARD_CONFIG['epsilon_nov'],
    reward_tn=REWARD_CONFIG['tn'],
    # --- FL Aggregation params ---
    aggregate_method=AGGREGATE_METHOD,
    fltrust_tau=FLTRUST_TAU,
    fltrust_beta=FLTRUST_BETA,
    fltrust_proxy_ratio=FLTRUST_PROXY_RATIO,
    fedplus_beta=FEDPLUS_BETA,
    fedplus_eta=FEDPLUS_ETA,
)

orchestrator.assign_data(agent_data, test_split_ratio=TEST_SPLIT_RATIO)

scenario_desc = {
    'A': 'NSL-KDD baseline',
    'B': 'CIC-IoMT-2024 same-domain',
    'C': 'CIC-IoMT-2024 -> CIC-Edge-IIoTSet-2022 cross-domain',
}[SCENARIO]

print(f'\nScenario {SCENARIO}: {scenario_desc}')
print(f'Agent: {AGENT_TYPE.upper()}, LR: {agent_lr}')
print(f'Aggregation: {AGGREGATE_METHOD}')
print(f'Starting training: {NUM_ROUNDS} rounds, {EPISODES_PER_ROUND} episodes/round ...')
start_time = time.time()

history = orchestrator.train(
    num_rounds=NUM_ROUNDS,
    global_test_X=X_test_final,
    global_test_y=y_test,
    verbose=True
)

total_time = time.time() - start_time
print(f'\nTraining complete in {total_time:.1f}s')

## Step 5 - Final Evaluation


In [ ]:
# ============================================================
# Cell 8 - Evaluation
# ============================================================
final_metrics = orchestrator.evaluate_global(X_test_final, y_test)

print('=' * 55)
if is_cross_dataset:
    print(f'  FINAL RESULTS (Train: {dataset_label}, Test: {test_label})')
else:
    print(f'  FINAL RESULTS ({dataset_label}, {EXPERIMENT} split)')
print('=' * 55)
print_metrics(final_metrics, 'Global Model')

if SCENARIO == 'A':
    print('\n--- Paper Table 3 (NSL-KDD Random) ---')
    print('Accuracy=0.9669  FPR=0.0195  Recall=0.9514')
    print('Precision=0.9769  F1=0.964  AUC=0.994')

## Step 6 - Visualization


In [ ]:
# ============================================================
# Cell 9 - Training curves & ROC
# ============================================================
import matplotlib
import matplotlib.pyplot as plt

if is_cross_dataset:
    plot_suffix = f'scenario_{SCENARIO}_{dataset_label}_to_{test_label}_{EXPERIMENT}_{AGENT_TYPE}_{AGGREGATE_METHOD}'
    title_suffix = f'(Train:{dataset_label}, Test:{test_label}, {EXPERIMENT}, {AGENT_TYPE.upper()}, {AGGREGATE_METHOD})'
else:
    plot_suffix = f'scenario_{SCENARIO}_{dataset_label}_{EXPERIMENT}_{AGENT_TYPE}_{AGGREGATE_METHOD}'
    title_suffix = f'({dataset_label}, {EXPERIMENT} split, {AGENT_TYPE.upper()}, {AGGREGATE_METHOD})'

In [ ]:
# ============================================================
# Cell 10 - Save results & global accuracy curve
# ============================================================
results = {
    'scenario': SCENARIO,
    'experiment': EXPERIMENT,
    'agent_type': AGENT_TYPE,
    'aggregate_method': AGGREGATE_METHOD,
    'dataset': dataset_label,
    'test_dataset': test_label,
    'cross_dataset': is_cross_dataset,
    'subsample_per_file': SUBSAMPLE_PER_FILE if SCENARIO != 'A' else None,
    'num_agents': num_agents,
    'num_rounds': NUM_ROUNDS,
    'attention_k': attn_k,
    'attention_a': attn_a,
    'reward_config': REWARD_CONFIG if AGENT_TYPE == 'ppo' else None,
    'final_metrics': {k: float(v) for k, v in final_metrics.items()},
    'training_time_sec': total_time,
}
results_path = os.path.join(OUTPUT_DIR, f'results_{plot_suffix}.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {results_path}')

---

## How to Run All 3 Scenarios

### Configuration (Cell 2)

| Variable     | Options                    | Description                                 |
| ------------ | -------------------------- | ------------------------------------------- |
| `SCENARIO`   | `'A'`, `'B'`, `'C'`        | Dataset scenario                            |
| `AGENT_TYPE` | `'dqn'`, `'ppo'`           | RL algorithm (paper original / improvement) |
| `EXPERIMENT` | `'random'`, `'customized'` | Data distribution strategy                  |

### DQN (bài báo gốc)

1. **Scenario A:** Set `SCENARIO = 'A'`, `AGENT_TYPE = 'dqn'`, Run All
2. **Scenario B:** Set `SCENARIO = 'B'`, `AGENT_TYPE = 'dqn'`, Run All
3. **Scenario C:** Set `SCENARIO = 'C'`, `AGENT_TYPE = 'dqn'`, Run All

### PPO (cải tiến)

1. **Scenario A:** Set `SCENARIO = 'A'`, `AGENT_TYPE = 'ppo'`, Run All
2. **Scenario B:** Set `SCENARIO = 'B'`, `AGENT_TYPE = 'ppo'`, Run All
3. **Scenario C:** Set `SCENARIO = 'C'`, `AGENT_TYPE = 'ppo'`, Run All

### Reward tuning (chỉ PPO)

Chỉnh `REWARD_CONFIG` trong Cell 2 để thay đổi trọng số reward:

- Tăng `gamma_fn` cho môi trường nhạy cảm (giảm FN)
- Giảm `beta` nếu cho phép nhiều FP hơn
- Tăng `epsilon_nov` để khuyến khích phát hiện attack patterns mới

Results will be saved to `/kaggle/working/results/` with separate plot folders for each scenario + agent type combination.
